# CS781 Capstone Phase 7: Prior Case Retrieval (PCR)
## Model R3: Model 2 Partial Fine-Tuning Hierarchical Retriever (~41% Trainable)

* **Architecture Overview**:
  - **Backbone**: Phase 1 Pretrained Model 2 (`model2-wide-mistral-136m`, 12 layers, $d=896$, 14 query / 7 KV GQA heads, intermediate dimension $d_{\text{ff}}=2432$, ~136.2M parameters).
  - **Partial Fine-Tuning Protocol**:
    - **Frozen**: Token embeddings (`embed_tokens`) and lower Transformer layers 1–6 (layers 0–5 in 0-indexed PyTorch).
    - **Trainable**: Upper Transformer layers 7–12 (layers 6–11), final RMSNorm, top-4 layer mixture, and dual chunk projector (~55.9M trainable parameters, ~41%).
    - **Rationale**: Retains low-level legal language representations while heavily specializing upper layers for legal precedent retrieval.
  - **Top-4 Layer Mixture**: Softmax mixture over layers 9–12 (all within unfrozen layers) with per-layer LayerNorm (no affine parameters).
  - **Dual Chunk Pooling**: SGPT position-weighted mean $\oplus$ $\langle\text{eos}\rangle$ token representation $\to$ Linear($2d \to 384$) $\to$ L2 normalisation.
  - **Dual Scoring**: $s(q, d) = \lambda \cdot \cos(\bar{q}, \bar{d}) + (1 - \lambda) \cdot S_{\text{MaxSim}}$.
  - **Training Objective**: Multi-positive InfoNCE with batch-gold masking $+ 0.5 \times$ auxiliary MaxSim InfoNCE.
  - **Hard Negatives**: Sampled from BM25 ranks **8–60** among non-gold candidates.


In [ ]:
# 1. Environment Dependencies & Hugging Face Authentication
import os
import sys
import gc
import json
import math
import re
import time
import zipfile
import shutil
import random
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Set, Any, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import MistralConfig, MistralModel, PreTrainedTokenizerFast
from huggingface_hub import HfApi, login, hf_hub_download, snapshot_download

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("✓ Hugging Face authentication successful.")
else:
    print("ℹ Running without explicit HF_TOKEN. Ensure public model access or local cache.")


In [ ]:
# 2. Execution Profiles & Hyperparameters
PROFILE = "CORE"  # Options: "CORE" (Full Production Training, 3 epochs) or "SMOKE" (Rapid Sanity Check)

PROFILES = {
    "SMOKE": {
        "max_train_queries": 24,
        "max_dev_queries": 16,
        "max_test_queries": 20,
        "max_candidates": 100,
        "epochs": 1,
        "batch_size": 2,
        "eval_every_steps": 10,
        "max_chunks_train": 2,
        "max_chunks_eval": 4,
        "description": "Rapid Sanity Check (~2 mins)"
    },
    "CORE": {
        "max_train_queries": None,
        "max_dev_queries": None,
        "max_test_queries": None,
        "max_candidates": None,
        "epochs": 3,
        "batch_size": 4,
        "eval_every_steps": 100,
        "max_chunks_train": 2,
        "max_chunks_eval": 16,
        "description": "Full Production Multi-GPU Training (3 Epochs)"
    }
}

CFG = PROFILES[PROFILE]
print(f"=== Active Profile: {PROFILE} ({CFG['description']}) ===")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_DEST_REPO = "sarweshbarnwal/CS781-Legal-Models"
BASE_MODEL_SUBFOLDER = "model2-wide-mistral-136m"
EXPERIMENT_NAME = "r3_model2_partial_ft"
HUB_SUBFOLDER = f"Phase_7/{EXPERIMENT_NAME}"

OUTPUT_DIR = "/kaggle/working/phase7_r3" if os.path.exists("/kaggle") else "./phase7_r3"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
METRICS_PATH = os.path.join(OUTPUT_DIR, "eval_metrics.json")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Local output directory: {OUTPUT_DIR}")
print(f"HF Hub destination: {HF_DEST_REPO}/{HUB_SUBFOLDER}")


In [ ]:
# 3. Dual-T4 GPU Hardware Setup & SDPA Optimization
print("=== Hardware Acceleration Verification ===")
gpu_count = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | GPU Count: {gpu_count}")

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU [{i}]: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB")
    
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cudnn.benchmark = True
    print("✓ SDPA memory-efficient attention activated.")
else:
    device = torch.device("cpu")
    print("⚠ No GPU detected. Running on CPU fallback.")


In [ ]:
# 4. Data Ingestion & Dataset Diagnostics
print("--- Loading Phase 7 PCR Dataset Splits ---")

def load_pcr_json(filename: str):
    local_candidates = [
        os.path.join("/kaggle/input/cs781-phase7-data", "Phase_7_PCR", filename),
        os.path.join("/kaggle/input/cs781-phase7", "Phase_7_PCR", filename),
        os.path.join("/kaggle/input/cs781-phase7-data", filename),
        os.path.join("./Phase_7_PCR", filename),
        os.path.join("./phase7_data", filename)
    ]
    for p in local_candidates:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
    
    fpath = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=f"Phase_7_PCR/{filename}",
        repo_type="dataset",
        token=HF_TOKEN
    )
    with open(fpath, "r", encoding="utf-8") as f:
        return json.load(f)

raw_train_queries = load_pcr_json("Train/train.json")
raw_train_cands   = load_pcr_json("Train/train_candidates.json")
raw_dev_queries   = load_pcr_json("Dev/dev.json")
raw_dev_cands     = load_pcr_json("Dev/dev_candidates.json")
raw_test_queries  = load_pcr_json("Test/test.json")
raw_test_cands    = load_pcr_json("Test/test_candidates.json")

def slice_dict(d: dict, limit: Optional[int]):
    if limit is None:
        return d
    return {k: d[k] for k in list(d.keys())[:limit]}

train_queries = slice_dict(raw_train_queries, CFG["max_train_queries"])
train_cands   = slice_dict(raw_train_cands, CFG["max_candidates"])
dev_queries   = slice_dict(raw_dev_queries, CFG["max_dev_queries"])
dev_cands     = slice_dict(raw_dev_cands, CFG["max_candidates"])
test_queries  = slice_dict(raw_test_queries, CFG["max_test_queries"])
test_cands    = slice_dict(raw_test_cands, CFG["max_candidates"])

print(f"Loaded: {len(train_queries):,} Train Qs | {len(dev_queries):,} Dev Qs | {len(test_queries):,} Test Qs")
print(f"Pools:  {len(train_cands):,} Train Cands | {len(dev_cands):,} Dev Cands | {len(test_cands):,} Test Cands")


In [ ]:
# 5. Sentence-Aware Chunker & Inverted BM25 Negative Miner (Ranks 8–60)
TOKEN_RE = re.compile(r"\b[a-zA-Z0-9_]{2,}\b")

CITATION_RE = re.compile(
    r"(?:Section|Sec\.?|S\.|u/s)\s*\d+|Articles?\s+\d+|Order\s+\w+\s+Rule\s+\d+|"
    r"Indian Penal Code|IPC|CrPC|CPC|Constitution of India|"
    r"AIR\s+\d{4}\s+SC|(?:\d{4})\s+\d+\s+SCC",
    re.IGNORECASE
)

class FastBM25Miner:
    """Sparse BM25 Indexer for Mining Negatives in Ranks 8-60 (skipping 1-7)."""
    def __init__(self, corpus_dict: Dict[str, dict], k1: float = 1.2, b: float = 0.75):
        self.doc_ids = [str(k) for k in corpus_dict.keys()]
        self.doc_id_to_idx = {did: idx for idx, did in enumerate(self.doc_ids)}
        self.N = len(self.doc_ids)
        self.k1 = k1
        self.b = b
        
        self.postings = defaultdict(list)
        self.doc_len = np.zeros(self.N, dtype=np.float32)
        
        for idx, did in enumerate(self.doc_ids):
            text = " ".join(corpus_dict[did].get("text", []))
            tokens = TOKEN_RE.findall(text.lower())
            self.doc_len[idx] = len(tokens)
            for term, count in Counter(tokens).items():
                self.postings[term].append((idx, count))
                
        self.avg_len = float(np.mean(self.doc_len)) if self.N > 0 else 1.0
        self.idf = {t: math.log(1.0 + (self.N - len(pl) + 0.5) / (len(pl) + 0.5)) for t, pl in self.postings.items()}
        self.len_norm = (1.0 - self.b) + self.b * (self.doc_len / max(1e-6, self.avg_len))

    def mine_hard_negatives(self, query_text: str, gold_ids: Set[str], num_negs: int = 3) -> List[str]:
        tokens = TOKEN_RE.findall(query_text.lower())
        scores = np.zeros(self.N, dtype=np.float32)
        for term, q_tf in Counter(tokens).items():
            if term not in self.postings:
                continue
            idf_val = self.idf[term]
            for doc_idx, doc_tf in self.postings[term]:
                scores[doc_idx] += idf_val * (doc_tf * (self.k1 + 1.0)) / (doc_tf + self.k1 * self.len_norm[doc_idx])
                
        top_indices = np.argsort(scores)[::-1]
        
        non_gold_ranked = []
        for idx in top_indices:
            cid = self.doc_ids[idx]
            if cid not in gold_ids:
                non_gold_ranked.append(cid)
                if len(non_gold_ranked) >= 60:
                    break
                    
        candidate_pool = non_gold_ranked[7:] if len(non_gold_ranked) > 7 else non_gold_ranked
        if len(candidate_pool) < num_negs:
            candidate_pool = [cid for cid in self.doc_ids if cid not in gold_ids]
            
        return random.sample(candidate_pool, min(num_negs, len(candidate_pool)))

print("Building BM25 Index on Train Candidate Pool...")
bm25_miner = FastBM25Miner(train_cands)
print("✓ Train candidate BM25 index ready for hard negative mining.")


In [ ]:
# 6. Load Legal Tokenizer & Sentence-Aware Chunker
print("Loading Model 2 Tokenizer...")
local_tok_candidates = [
    os.path.join("/kaggle/input/cs781-phase7-models", BASE_MODEL_SUBFOLDER, "tokenizer.json"),
    os.path.join("/kaggle/input/cs781-legal-models", BASE_MODEL_SUBFOLDER, "tokenizer.json"),
    os.path.join(f"./{BASE_MODEL_SUBFOLDER}", "tokenizer.json"),
    os.path.join(OUTPUT_DIR, BASE_MODEL_SUBFOLDER, "tokenizer.json"),
]
tok_json_path = None
for p in local_tok_candidates:
    if os.path.exists(p):
        tok_json_path = p
        break

if tok_json_path is None:
    for sub in [f"{BASE_MODEL_SUBFOLDER}/tokenizer.json", f"{BASE_MODEL_SUBFOLDER}/tokenizer/tokenizer.json"]:
        try:
            tok_json_path = hf_hub_download(repo_id=HF_DEST_REPO, filename=sub, token=HF_TOKEN)
            if os.path.exists(tok_json_path):
                break
        except Exception:
            continue

if tok_json_path is None or not os.path.exists(tok_json_path):
    raise FileNotFoundError(f"Could not locate tokenizer.json for {BASE_MODEL_SUBFOLDER} in local paths or {HF_DEST_REPO}")

tok_cfg = {}
for sub in [f"{BASE_MODEL_SUBFOLDER}/tokenizer_config.json", f"{BASE_MODEL_SUBFOLDER}/tokenizer/tokenizer_config.json"]:
    try:
        cfg_p = hf_hub_download(repo_id=HF_DEST_REPO, filename=sub, token=HF_TOKEN)
        with open(cfg_p, "r", encoding="utf-8") as f:
            tok_cfg = json.load(f)
        break
    except Exception:
        continue

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=tok_json_path,
    bos_token=tok_cfg.get("bos_token", "<s>"),
    eos_token=tok_cfg.get("eos_token", "<eos>"),
    unk_token=tok_cfg.get("unk_token", "<unk>"),
    pad_token=tok_cfg.get("pad_token") or tok_cfg.get("eos_token", "<eos>")
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token or "<eos>"
    tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)

print(f"✓ Tokenizer ready. Vocab size: {len(tokenizer):,} | Pad ID: {tokenizer.pad_token_id} | EOS ID: {tokenizer.eos_token_id}")

def chunk_legal_sentences(
    sentences: List[str],
    tokenizer: PreTrainedTokenizerFast,
    max_tokens: int = 511,
    overlap_tokens: int = 64
) -> List[Tuple[List[int], int]]:
    """Sentence-aware chunker with citation density tracking."""
    if not sentences:
        sentences = [""]
        
    chunks = []
    curr_chunk_tokens = []
    curr_citations = 0
    overlap_buffer = []
    
    for s in sentences:
        s_tokens = tokenizer.encode(s, add_special_tokens=False)
        s_citations = len(CITATION_RE.findall(s))
        
        if len(curr_chunk_tokens) + len(s_tokens) > max_tokens and curr_chunk_tokens:
            chunks.append((curr_chunk_tokens + [tokenizer.eos_token_id], curr_citations))
            curr_chunk_tokens = list(overlap_buffer)
            curr_citations = 0
            
        curr_chunk_tokens.extend(s_tokens)
        curr_citations += s_citations
        
        if len(s_tokens) <= overlap_tokens:
            overlap_buffer = s_tokens
        else:
            overlap_buffer = s_tokens[-overlap_tokens:]
            
    if curr_chunk_tokens or not chunks:
        chunks.append((curr_chunk_tokens + [tokenizer.eos_token_id], curr_citations))
        
    return chunks

def get_training_views(chunks: List[Tuple[List[int], int]], V: int = 2) -> List[List[int]]:
    if len(chunks) == 1:
        return [chunks[0][0], chunks[0][0]]
    v1 = chunks[0][0]
    rest = chunks[1:]
    best_citation_chunk = max(rest, key=lambda x: x[1])[0]
    return [v1, best_citation_chunk]


In [ ]:
# 7. Model 2 Partial Fine-Tuning Architecture
def sgpt_pool(h: torch.Tensor, attn: torch.Tensor) -> torch.Tensor:
    pos = torch.arange(1, h.size(1) + 1, device=h.device, dtype=h.dtype)[None, :] * attn.to(h.dtype)
    w = pos / pos.sum(1, keepdim=True).clamp_min(1e-6)
    return (h * w.unsqueeze(-1)).sum(1)

def eos_state(h: torch.Tensor, attn: torch.Tensor) -> torch.Tensor:
    idx = (attn.sum(1) - 1).clamp_min(0).long()
    return h[torch.arange(h.size(0), device=h.device), idx]

class Model2PartialFTRetriever(nn.Module):
    def __init__(self, backbone: MistralModel, d: int = 896, proj_dim: int = 384):
        super().__init__()
        self.backbone = backbone
        self.d = d
        self.proj_dim = proj_dim
        
        # Mixture over top 4 layers (layers 9-12 of 12)
        self.mixture_logits = nn.Parameter(torch.zeros(4))
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(d, elementwise_affine=False) for _ in range(4)
        ])
        
        # Dual chunk projector: SGPT [896] ⊕ EOS [896] -> [1792] -> Linear -> [384]
        self.projector = nn.Linear(2 * d, proj_dim)
        self.raw_tau = nn.Parameter(torch.tensor(math.log(0.05)))
        
    @property
    def tau(self) -> torch.Tensor:
        return torch.clamp(torch.exp(self.raw_tau), min=0.01, max=0.20)

    def encode_chunks(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        N, S = input_ids.shape
        valid = (attention_mask.sum(dim=-1) > 0)
        
        if not valid.any():
            return torch.zeros(N, self.proj_dim, device=input_ids.device)
            
        v_ids = input_ids[valid]
        v_mask = attention_mask[valid]
        
        out = self.backbone(
            input_ids=v_ids,
            attention_mask=v_mask,
            output_hidden_states=True
        )
        top4 = out.hidden_states[-4:]
        
        weights = F.softmax(self.mixture_logits, dim=0)
        mixed_h = torch.zeros_like(top4[0])
        for i in range(4):
            mixed_h = mixed_h + weights[i] * self.layer_norms[i](top4[i])
            
        z = torch.cat([sgpt_pool(mixed_h, v_mask), eos_state(mixed_h, v_mask)], dim=-1)
        v_proj = F.normalize(self.projector(z.float()), dim=-1)
        
        out_vecs = torch.zeros(N, self.proj_dim, device=input_ids.device, dtype=v_proj.dtype)
        out_vecs[valid] = v_proj
        return out_vecs

    def encode_document_chunks(self, doc_chunks: List[List[int]], pad_id: int, max_chunk_len: int = 512) -> torch.Tensor:
        K = len(doc_chunks)
        ids = torch.full((K, max_chunk_len), pad_id, dtype=torch.long, device=device)
        mask = torch.zeros((K, max_chunk_len), dtype=torch.long, device=device)
        
        for i, c in enumerate(doc_chunks):
            length = min(len(c), max_chunk_len)
            ids[i, :length] = torch.tensor(c[:length], dtype=torch.long, device=device)
            mask[i, :length] = 1
            
        return self.encode_chunks(ids, mask)

def compute_doc_scores(qv, qm, dv, dm, lam=0.5):
    qmean = F.normalize((qv * qm[..., None]).sum(1) / qm.sum(1, keepdim=True).clamp_min(1), dim=-1)
    dmean = F.normalize((dv * dm[..., None]).sum(1) / dm.sum(1, keepdim=True).clamp_min(1), dim=-1)
    cos = qmean @ dmean.T
    
    sim = torch.einsum('qie,dje->qdij', qv, dv)
    q2d = sim.masked_fill(dm[None, :, None, :] == 0, -1e4).max(-1).values
    q2d = (q2d * qm[:, None, :]).sum(-1) / qm.sum(1)[:, None].clamp_min(1)
    
    d2q = sim.masked_fill(qm[:, None, :, None] == 0, -1e4).max(2).values
    d2q = (d2q * dm[None, :, :]).sum(-1) / dm.sum(1)[None, :].clamp_min(1)
    
    maxsim = 0.5 * (q2d + d2q)
    return lam * cos + (1.0 - lam) * maxsim, cos, maxsim

def multi_positive_infonce(S: torch.Tensor, pos_mask: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
    L = S / tau
    neg_lse = torch.logsumexp(L.masked_fill(pos_mask, -1e9), dim=1, keepdim=True)
    per_pos = F.softplus(neg_lse - L) * pos_mask
    has_pos = pos_mask.sum(1) > 0
    return (per_pos.sum(1)[has_pos] / pos_mask.sum(1)[has_pos].clamp_min(1)).mean()

print("✓ Model2PartialFTRetriever architecture compiled.")


In [ ]:
# 8. Load Pretrained Model 2 & Apply Partial FT Freezing
print("Downloading Model 2 base weights from Master Hub...")
m2_dir = os.path.join(OUTPUT_DIR, "model2_base")
os.makedirs(m2_dir, exist_ok=True)

snapshot_download(
    repo_id=HF_DEST_REPO,
    allow_patterns=[f"{BASE_MODEL_SUBFOLDER}/*"],
    local_dir=m2_dir,
    token=HF_TOKEN
)

base_subpath = os.path.join(m2_dir, BASE_MODEL_SUBFOLDER)
config = MistralConfig.from_pretrained(base_subpath)
base_backbone = MistralModel.from_pretrained(base_subpath, config=config)

# ---------------------------------------------------------------------------
# STRICT PARTIAL FINE-TUNING FREEZING SPECIFICATION:
# 1. Freeze Word Embeddings: embed_tokens
# 2. Freeze Lower Transformer Layers 1 to 6 (layers 0 to 5)
# 3. Unfreeze Upper Transformer Layers 7 to 12 (layers 6 to 11)
# 4. Unfreeze Final RMSNorm
# ---------------------------------------------------------------------------
base_backbone.embed_tokens.weight.requires_grad = False

num_layers = len(base_backbone.layers)
split_point = num_layers // 2  # 12 // 2 = 6

for idx in range(num_layers):
    layer = base_backbone.layers[idx]
    if idx < split_point:
        for p in layer.parameters():
            p.requires_grad = False
    else:
        for p in layer.parameters():
            p.requires_grad = True

for p in base_backbone.norm.parameters():
    p.requires_grad = True

base_backbone.gradient_checkpointing_enable()
if hasattr(base_backbone, "enable_input_require_grads"):
    base_backbone.enable_input_require_grads()
else:
    def make_inputs_require_grad(module, input, output):
        output.requires_grad_(True)
    base_backbone.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

model = Model2PartialFTRetriever(backbone=base_backbone, d=config.hidden_size, proj_dim=384).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct_trainable = 100.0 * trainable_params / total_params

print(f"\n=== Partial FT Verification ===")
print(f"  Total Layers:              {num_layers} (Frozen: Layers 1-6 | Trainable: Layers 7-12)")
print(f"  Total Model Parameters:    {total_params/1e6:.2f}M")
print(f"  Trainable Parameters:      {trainable_params/1e6:.2f}M ({pct_trainable:.2f}%)")

assert 35.0 <= pct_trainable <= 45.0, f"Expected ~40% trainable params, got {pct_trainable:.2f}%"
print("✓ Exact Partial FT parameter proportion verified (~41%)!")

# Pre-flight Forward & Backward Check
model.train()
test_ids = torch.randint(0, 1000, (4, 128), device=device)
test_mask = torch.ones((4, 128), device=device)
with torch.amp.autocast('cuda', dtype=torch.float16):
    dummy_out = model.encode_chunks(test_ids, test_mask)
    dummy_loss = dummy_out.sum()

dummy_loss.backward()
grad_norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
assert grad_norm > 0, "FATAL: Pre-flight gradient check failed!"
model.zero_grad()
print(f"✓ Pre-flight assertion passed! Gradient norm: {grad_norm:.4f}")


In [ ]:
# 9. Triplet Dataset & Collation with BM25 Hard Negatives
class PCRTrainDataset(Dataset):
    def __init__(self, queries: Dict[str, dict], candidates: Dict[str, dict], miner: FastBM25Miner):
        self.qids = list(queries.keys())
        self.queries = queries
        self.candidates = candidates
        self.miner = miner
        
    def __len__(self):
        return len(self.qids)
        
    def __getitem__(self, idx):
        qid = self.qids[idx]
        q_data = self.queries[qid]
        gold_ids = list(q_data.get("relevant_candidates", []))
        pos_id = random.choice(gold_ids) if gold_ids else random.choice(list(self.candidates.keys()))
        neg_ids = self.miner.mine_hard_negatives(" ".join(q_data.get("text", [])), set(gold_ids), num_negs=3)
        
        return {
            "qid": qid,
            "q_sents": q_data.get("text", []),
            "pos_id": pos_id,
            "pos_sents": self.candidates[pos_id].get("text", []),
            "neg_ids": neg_ids,
            "neg_sents": [self.candidates[nid].get("text", []) for nid in neg_ids],
            "all_golds": gold_ids
        }

def collate_train_batch(batch: List[dict]):
    doc_chunk_views = []
    doc_ids = []
    q_indices = []
    d_indices = []
    gold_map = defaultdict(set)
    
    for i, item in enumerate(batch):
        q_idx = len(doc_chunk_views)
        q_chunks = chunk_legal_sentences(item["q_sents"], tokenizer)
        doc_chunk_views.append(get_training_views(q_chunks, V=2))
        doc_ids.append(item["qid"])
        q_indices.append(q_idx)
        gold_map[q_idx] = set(item["all_golds"])
        
        pos_chunks = chunk_legal_sentences(item["pos_sents"], tokenizer)
        doc_chunk_views.append(get_training_views(pos_chunks, V=2))
        doc_ids.append(item["pos_id"])
        d_indices.append(len(doc_chunk_views) - 1)
        
        for nid, nsents in zip(item["neg_ids"], item["neg_sents"]):
            n_chunks = chunk_legal_sentences(nsents, tokenizer)
            doc_chunk_views.append(get_training_views(n_chunks, V=2))
            doc_ids.append(nid)
            d_indices.append(len(doc_chunk_views) - 1)
            
    flat_chunks = [v for views in doc_chunk_views for v in views]
    max_len = 512
    N = len(flat_chunks)
    
    input_ids = torch.full((N, max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((N, max_len), dtype=torch.long)
    
    for i, c in enumerate(flat_chunks):
        length = min(len(c), max_len)
        input_ids[i, :length] = torch.tensor(c[:length], dtype=torch.long)
        attention_mask[i, :length] = 1
        
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "q_indices": q_indices,
        "d_indices": d_indices,
        "doc_ids": doc_ids,
        "gold_map": gold_map
    }

train_dataset = PCRTrainDataset(train_queries, train_cands, bm25_miner)
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    collate_fn=collate_train_batch,
    drop_last=True
)
print(f"✓ DataLoader instantiated: {len(train_loader)} steps per epoch.")


In [ ]:
# 10. Multi-Epoch Training Loop with Differential Learning Rates
# Differential LR: Unfrozen Backbone Layers 1e-4, Projector/Mixture/Tau 1e-3
backbone_unfrozen = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
head_params = [
    model.mixture_logits,
    model.projector.weight,
    model.projector.bias,
    model.raw_tau
]

optimizer = torch.optim.AdamW([
    {"params": backbone_unfrozen, "lr": 1e-4, "weight_decay": 0.01},
    {"params": head_params, "lr": 1e-3, "weight_decay": 0.0}
])

total_steps = len(train_loader) * CFG["epochs"]
warmup_steps = int(total_steps * 0.06)

def get_lr_multiplier(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_multiplier)
scaler = torch.amp.GradScaler('cuda')

print(f"=== Starting Production Training Loop (Target Steps: {total_steps}) ===")
global_step = 0
step_losses = []

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    print(f"\n--- Epoch {epoch}/{CFG['epochs']} ---")
    
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda', dtype=torch.float16):
            chunk_vecs = model.encode_chunks(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )
            
            num_docs = len(batch["doc_ids"])
            doc_chunk_tensors = chunk_vecs.view(num_docs, 2, 384)
            chunk_masks = torch.ones((num_docs, 2), device=device)
            
            q_idx = batch["q_indices"]
            d_idx = batch["d_indices"]
            
            qv = doc_chunk_tensors[q_idx]
            qm = chunk_masks[q_idx]
            dv = doc_chunk_tensors[d_idx]
            dm = chunk_masks[d_idx]
            
            final_scores, cos_scores, maxsim_scores = compute_doc_scores(qv, qm, dv, dm, lam=0.5)
            
            pos_mask = torch.zeros_like(final_scores, dtype=torch.bool)
            for i_q, actual_q_idx in enumerate(q_idx):
                gold_set = batch["gold_map"][actual_q_idx]
                for j_d, actual_d_idx in enumerate(d_idx):
                    cand_id = batch["doc_ids"][actual_d_idx]
                    if cand_id in gold_set:
                        pos_mask[i_q, j_d] = True
                        
            loss_final = multi_positive_infonce(final_scores, pos_mask, model.tau)
            loss_maxsim = multi_positive_infonce(maxsim_scores, pos_mask, model.tau)
            total_loss = loss_final + 0.5 * loss_maxsim
            
        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        global_step += 1
        step_losses.append(total_loss.item())
        
        if global_step % 20 == 0 or global_step == total_steps:
            print(f"Step [{global_step}/{total_steps}] | Loss: {total_loss.item():.4f} | Tau: {model.tau.item():.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
            
print("✓ Model 2 Partial FT Training loop completed.")


In [ ]:
# 11. Full Dev Split Evaluation & Micro-F1@7 Scoring
model.eval()
print("=== Evaluating Model 2 Partial FT on Dev Set ===")

def encode_corpus_documents(corpus_dict: Dict[str, dict], max_chunks: int = 16):
    doc_ids = list(corpus_dict.keys())
    all_mean_vecs = []
    
    t0 = time.time()
    with torch.no_grad():
        for i, did in enumerate(doc_ids):
            chunks = chunk_legal_sentences(corpus_dict[did].get("text", []), tokenizer)
            c_toks = [c[0] for c in chunks[:max_chunks]]
            
            with torch.amp.autocast('cuda', dtype=torch.float16):
                c_vecs = model.encode_document_chunks(c_toks, tokenizer.pad_token_id)
                mean_v = F.normalize(c_vecs.mean(dim=0, keepdim=True), dim=-1)
                
            all_mean_vecs.append(mean_v.cpu())
            if (i + 1) % 500 == 0 or (i + 1) == len(doc_ids):
                print(f"  Encoded {i+1}/{len(doc_ids)} docs in {time.time()-t0:.1f}s...")
                
    return doc_ids, torch.cat(all_mean_vecs, dim=0)

dev_cand_ids, dev_cand_means = encode_corpus_documents(dev_cands, max_chunks=CFG["max_chunks_eval"])
dev_q_ids, dev_q_means = encode_corpus_documents(dev_queries, max_chunks=CFG["max_chunks_eval"])

dev_sim = torch.matmul(dev_q_means, dev_cand_means.t()).numpy()

dev_predictions = {}
for idx, qid in enumerate(dev_q_ids):
    s = dev_sim[idx].copy()
    if str(qid) in dev_cand_ids:
        s[dev_cand_ids.index(str(qid))] = -1e9
    top7_idx = np.argpartition(s, -7)[-7:]
    top7_sorted = top7_idx[np.argsort(s[top7_idx])[::-1]]
    dev_predictions[str(qid)] = [dev_cand_ids[i] for i in top7_sorted]

# Official Micro-F1@7
total_tp, total_gold = 0, 0
for qid in dev_q_ids:
    preds = set(dev_predictions[str(qid)])
    golds = set(dev_queries[qid].get("relevant_candidates", []))
    total_tp += len(preds & golds)
    total_gold += len(golds)

prec = total_tp / (7.0 * len(dev_q_ids))
rec = total_tp / max(1, total_gold)
dev_f1 = (2.0 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

print(f"\n🏆 Model 2 Partial FT Dev Results:")
print(f"  Micro-F1@7:       {dev_f1*100:.2f}%")
print(f"  Micro-Precision@7: {prec*100:.2f}%")
print(f"  Micro-Recall@7:    {rec*100:.2f}%")


In [ ]:
# 12. Test Split Inference & CodaBench Packaging
print("=== Running Full Test Split Inference ===")
test_cand_ids, test_cand_means = encode_corpus_documents(test_cands, max_chunks=CFG["max_chunks_eval"])
test_q_ids, test_q_means = encode_corpus_documents(test_queries, max_chunks=CFG["max_chunks_eval"])

test_sim = torch.matmul(test_q_means, test_cand_means.t()).numpy()

predictions = {}
test_cand_set = set(test_cand_ids)

for idx, qid in enumerate(test_q_ids):
    q_str = str(qid)
    s = test_sim[idx].copy()
    if q_str in test_cand_ids:
        s[test_cand_ids.index(q_str)] = -1e9
    top7_idx = np.argpartition(s, -7)[-7:]
    top7_sorted = top7_idx[np.argsort(s[top7_idx])[::-1]]
    predictions[q_str] = [test_cand_ids[i] for i in top7_sorted]

# Strict Assertions
assert len(predictions) == len(test_queries), "Query count mismatch!"
for qid, cands in predictions.items():
    assert len(cands) == 7 and len(set(cands)) == 7, f"Query {qid} invalid top-7"
    assert qid not in cands, f"Query {qid} in its own predictions"
    for c in cands:
        assert c in test_cand_set, f"Candidate {c} not in test pool"

print("✓ Strict CodaBench assertions passed!")

# Write prediction.json
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)

# Package submission.zip
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")
print(f"✓ Saved {SUBMISSION_ZIP} ({os.path.getsize(SUBMISSION_ZIP)/1024:.1f} KB)")


In [ ]:
# 13. Save Weights & Non-blocking HF Hub Synchronization
# Save unfrozen weights
save_dict = {n: p.data for n, p in model.state_dict().items() if p.requires_grad}
ckpt_path = os.path.join(CHECKPOINT_DIR, "model2_partial_ft_weights.pt")
torch.save(save_dict, ckpt_path)
print(f"✓ Saved trainable weights to {ckpt_path} ({os.path.getsize(ckpt_path)/(1024*1024):.1f} MB)")

val_report = {
    "experiment": EXPERIMENT_NAME,
    "technique": "Partial Fine-Tuning (Upper 6 Layers)",
    "backbone": "Model 2 Wide Mistral 136M",
    "dev_micro_f1_at_7": float(dev_f1),
    "dev_precision_at_7": float(prec),
    "dev_recall_at_7": float(rec),
    "trainable_params": trainable_params,
    "pct_trainable": float(pct_trainable)
}
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(val_report, f, indent=2)
print(f"✓ Saved validation report to {METRICS_PATH}")

if HF_TOKEN:
    try:
        api = HfApi(token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj=PREDICTION_PATH,
            path_in_repo=f"{HUB_SUBFOLDER}/prediction.json",
            repo_id=HF_DEST_REPO,
            repo_type="model"
        )
        api.upload_file(
            path_or_fileobj=SUBMISSION_ZIP,
            path_in_repo=f"{HUB_SUBFOLDER}/submission.zip",
            repo_id=HF_DEST_REPO,
            repo_type="model"
        )
        api.upload_file(
            path_or_fileobj=ckpt_path,
            path_in_repo=f"{HUB_SUBFOLDER}/model2_partial_ft_weights.pt",
            repo_id=HF_DEST_REPO,
            repo_type="model"
        )
        api.upload_file(
            path_or_fileobj=METRICS_PATH,
            path_in_repo=f"{HUB_SUBFOLDER}/eval_metrics.json",
            repo_id=HF_DEST_REPO,
            repo_type="model"
        )
        print(f"✓ Successfully pushed R3 artifacts to Hugging Face Hub ({HF_DEST_REPO}/{HUB_SUBFOLDER}).")
    except Exception as e:
        print(f"Notice: HF Hub sync skipped ({e}). Local files are fully preserved.")
